# Tache 1 - v15 DERNIERE SOUMISSION - Chunks + Two-Pass Test

**v14 resultat** : AUC=96.76 (leaderboard) vs 99.75 (val)
**Probleme** : en test, les fenetres de contexte melangent C et M (on ne connait pas les labels)
En train, les chunks etaient purs (meme label) → le modele s'attend a du texte coherent

**Solution v15 : Two-Pass**
1. Pass 1 : predire chaque phrase SEULE (pas de contexte)
2. Utiliser ces predictions pour grouper les phrases de meme prediction
3. Pass 2 : re-predire avec des fenetres de contexte "propres" (voisins de meme prediction)
4. Moyenne ponderee des deux passes


## 1. Installation

In [ ]:
!pip install -q transformers datasets accelerate scikit-learn torch

## 2. Imports

In [ ]:
import os, re, codecs, gc
import numpy as np
import torch
from torch.utils.data import Dataset as TorchDataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)
from sklearn.metrics import f1_score, average_precision_score, roc_auc_score
from sklearn.model_selection import train_test_split
from datasets import Dataset as HFDataset
from collections import defaultdict

os.environ["WANDB_DISABLED"] = "true"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

## 3. Chargement

In [ ]:
train_entries = []
with codecs.open("corpus_tache1_learn.utf8", "r", "utf-8") as f:
    for line in f:
        line = line.strip()
        if len(line) < 5:
            continue
        match = re.match(r"<(\d+):(\d+):([CM])>\s*(.*)", line)
        if match:
            train_entries.append({
                'doc': int(match.group(1)),
                'sent': int(match.group(2)),
                'label': match.group(3),
                'text': match.group(4),
            })

test_entries = []
with codecs.open("corpus_tache1_test.utf8", "r", "utf-8") as f:
    for line in f:
        line = line.strip()
        if len(line) < 5:
            continue
        match = re.match(r"<(\d+):(\d+)>\s*(.*)", line)
        if match:
            test_entries.append({
                'doc': int(match.group(1)),
                'sent': int(match.group(2)),
                'text': match.group(3),
            })

print(f"Train: {len(train_entries)} | Test: {len(test_entries)}")

## 4. Chunks TRAIN (meme que v14)

In [ ]:
MAX_WORDS = 350

chunks_train = []
current_doc = None
current_label = None
current_texts = []
current_words = 0

for entry in train_entries:
    words = len(entry['text'].split())
    if (entry['doc'] == current_doc and 
        entry['label'] == current_label and 
        current_words + words <= MAX_WORDS):
        current_texts.append(entry['text'])
        current_words += words
    else:
        if current_texts:
            chunks_train.append({
                'text': " ".join(current_texts),
                'label': 1 if current_label == 'M' else 0,
            })
        current_doc = entry['doc']
        current_label = entry['label']
        current_texts = [entry['text']]
        current_words = words

if current_texts:
    chunks_train.append({
        'text': " ".join(current_texts),
        'label': 1 if current_label == 'M' else 0,
    })

print(f"Train chunks: {len(chunks_train)}")

chunk_texts = [c['text'] for c in chunks_train]
chunk_labels = np.array([c['label'] for c in chunks_train])

train_texts, val_texts, train_labels, val_labels = train_test_split(
    chunk_texts, chunk_labels, test_size=0.05, random_state=42, stratify=chunk_labels
)
print(f"Train: {len(train_texts)} | Val: {len(val_texts)}")

## 5. Tokenizer + Dataset + Training (meme que v14)

In [ ]:
MODEL_NAME = "camembert/camembert-large"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class ChunkDataset(TorchDataset):
    def __init__(self, texts, labels, tokenizer, max_length=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx], padding="max_length",
            truncation=True, max_length=self.max_length,
            return_tensors="pt",
        )
        return {
            "input_ids": enc["input_ids"].squeeze(),
            "attention_mask": enc["attention_mask"].squeeze(),
            "label": torch.tensor(self.labels[idx], dtype=torch.long),
        }
    def __len__(self):
        return len(self.labels)

train_dataset = ChunkDataset(train_texts, train_labels, tokenizer)
val_dataset = ChunkDataset(val_texts, val_labels, tokenizer)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    p = torch.nn.functional.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()
    preds = (p > 0.5).astype(int)
    f1m = f1_score(labels, preds, average="macro")
    auc = roc_auc_score(labels, p) if len(np.unique(labels)) > 1 else 0
    ap = average_precision_score(labels, p) if len(np.unique(labels)) > 1 else 0
    print(f"  F1m={f1m*100:.1f} AUC={auc*100:.1f} AP={ap*100:.1f}")
    return {"f1_macro": round(f1m*100,3), "auc": round(auc*100,3), "ap": round(ap*100,3)}

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.to(device)

training_args = TrainingArguments(
    output_dir="./results_v15",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-5,
    num_train_epochs=10,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=4,
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="auc",
    greater_is_better=True,
    report_to="none",
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model, args=training_args,
    train_dataset=train_dataset, eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

## 6. PASS 1 : Predire chaque phrase TEST individuellement

In [ ]:
def tokenize_fn(ex):
    return tokenizer(ex["text"], padding="max_length", truncation=True, max_length=512)

# Predict each sentence alone
test_texts_solo = [e['text'] for e in test_entries]
test_hf_solo = HFDataset.from_dict({"text": test_texts_solo})
test_tok_solo = test_hf_solo.map(tokenize_fn, batched=True, batch_size=128)
preds_solo = trainer.predict(test_tok_solo)
p_solo = torch.nn.functional.softmax(torch.from_numpy(preds_solo.predictions), dim=-1)[:, 1].numpy()

print(f"Pass 1 (solo): {len(p_solo)} predictions")
print(f"  M predicted: {sum(p_solo > 0.5)} ({sum(p_solo > 0.5)/len(p_solo)*100:.1f}%)")

## 7. PASS 2 : Fenetres de contexte PROPRES

On utilise les predictions du Pass 1 pour grouper les phrases voisines qui ont la meme prediction. Les fenetres ne melangent plus C et M.

In [ ]:
test_by_doc = defaultdict(list)
for i, entry in enumerate(test_entries):
    test_by_doc[entry['doc']].append((i, entry['text'], p_solo[i]))

MAX_CONTEXT_WORDS = 350
test_windows_clean = []

for doc_id in sorted(test_by_doc.keys()):
    sents = test_by_doc[doc_id]  # (global_idx, text, solo_prob)
    
    for pos, (global_idx, text, prob) in enumerate(sents):
        predicted_label = 1 if prob > 0.5 else 0
        
        window = [text]
        word_count = len(text.split())
        
        left = pos - 1
        right = pos + 1
        
        while word_count < MAX_CONTEXT_WORDS:
            added = False
            # Only add neighbors with SAME predicted label
            if left >= 0:
                neighbor_pred = 1 if sents[left][2] > 0.5 else 0
                if neighbor_pred == predicted_label:
                    left_words = len(sents[left][1].split())
                    if word_count + left_words <= MAX_CONTEXT_WORDS:
                        window.insert(0, sents[left][1])
                        word_count += left_words
                        left -= 1
                        added = True
                    else:
                        left = -1  # stop
                else:
                    left = -1  # different prediction, stop
            
            if right < len(sents):
                neighbor_pred = 1 if sents[right][2] > 0.5 else 0
                if neighbor_pred == predicted_label:
                    right_words = len(sents[right][1].split())
                    if word_count + right_words <= MAX_CONTEXT_WORDS:
                        window.append(sents[right][1])
                        word_count += right_words
                        right += 1
                        added = True
                    else:
                        right = len(sents)  # stop
                else:
                    right = len(sents)  # different prediction, stop
            
            if not added:
                break
        
        test_windows_clean.append((global_idx, " ".join(window)))

test_windows_clean.sort(key=lambda x: x[0])
test_clean_texts = [w[1] for w in test_windows_clean]

print(f"Pass 2 windows: {len(test_clean_texts)}")
print(f"Avg words/window: {np.mean([len(t.split()) for t in test_clean_texts]):.0f}")

In [ ]:
# Predict with clean windows
test_hf_clean = HFDataset.from_dict({"text": test_clean_texts})
test_tok_clean = test_hf_clean.map(tokenize_fn, batched=True, batch_size=128)
preds_clean = trainer.predict(test_tok_clean)
p_clean = torch.nn.functional.softmax(torch.from_numpy(preds_clean.predictions), dim=-1)[:, 1].numpy()

print(f"Pass 2 (clean windows): {len(p_clean)} predictions")
print(f"  M predicted: {sum(p_clean > 0.5)} ({sum(p_clean > 0.5)/len(p_clean)*100:.1f}%)")

## 8. Combiner Pass 1 + Pass 2

In [ ]:
# Moyenne ponderee : plus de poids au pass 2 (contexte propre)
for alpha in [0.0, 0.2, 0.3, 0.5, 0.7, 0.8, 1.0]:
    p_combined = alpha * p_clean + (1 - alpha) * p_solo
    n_m = sum(p_combined > 0.5)
    print(f"alpha={alpha:.1f} (pass2={alpha:.0%} solo={1-alpha:.0%}): M={n_m} ({n_m/len(p_combined)*100:.1f}%)")

# Best is probably alpha=0.7 or 0.8 (favor clean context)
p_final = 0.7 * p_clean + 0.3 * p_solo

print(f"\nFinal: alpha=0.7")
print(f"  < 0.1:   {sum(p_final < 0.1):>6} ({sum(p_final < 0.1)/len(p_final)*100:.1f}%)")
print(f"  > 0.9:   {sum(p_final > 0.9):>6} ({sum(p_final > 0.9)/len(p_final)*100:.1f}%)")
print(f"  0.1-0.9: {sum((p_final >= 0.1) & (p_final <= 0.9)):>6} ({sum((p_final >= 0.1) & (p_final <= 0.9))/len(p_final)*100:.1f}%)")
print(f"\nCible ami: <0.1=85.8%, >0.9=11.4%, entre=2.8%")

## 9. Soumission CSV

In [ ]:
# Version principale : ensemble pass1+pass2
with open("predictions.csv", "w") as f:
    for p in p_final:
        f.write(f"{p}\n")
print(f"predictions.csv (ensemble 0.7*pass2 + 0.3*solo)")

# Version pass2 seul
with open("predictions_pass2.csv", "w") as f:
    for p in p_clean:
        f.write(f"{p}\n")
print(f"predictions_pass2.csv (clean windows seul)")

# Version pass1 seul
with open("predictions_solo.csv", "w") as f:
    for p in p_solo:
        f.write(f"{p}\n")
print(f"predictions_solo.csv (phrases seules)")

print()
print("SOUMETS predictions_pass2.csv — c'est les fenetres propres")

## 10. Nettoyage

In [ ]:
del model, trainer
gc.collect()
torch.cuda.empty_cache()